In [ ]:
# %% [markdown]
# # Tier-wise Net NPA Analysis
# This notebook loads the cleaned bank dataset, classifies banks into three tiers (large, medium, small) based on total net profit, and plots the average Net NPA (%) for each tier across years.

# %%
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# %%
# Load the dataset
df = pd.read_csv('cleaned_masterdataset.csv')

# Display basic info
print(df.head())
print(df['BANK'].unique())

# %%
# Clean bank names: strip extra spaces and standardize
df['BANK'] = df['BANK'].str.strip()

# Check unique banks again
banks = df['BANK'].unique()
print("Banks in dataset:", banks)

# %%
# Extract start year from the 'Year' column for proper sorting
def get_start_year(year_str):
    # Handle formats like '2005-2006', '2005-06', '2005-06*' etc.
    import re
    match = re.search(r'(\d{4})', year_str)
    if match:
        return int(match.group(1))
    else:
        return np.nan

df['StartYear'] = df['Year'].apply(get_start_year)
# Drop rows where year could not be parsed (if any)
df = df.dropna(subset=['StartYear'])
df['StartYear'] = df['StartYear'].astype(int)

# Sort by bank and year for consistency
df = df.sort_values(['BANK', 'StartYear']).reset_index(drop=True)

# %%
# Define tiers based on total net profit across all years
total_profit = df.groupby('BANK')['Net Profit (cr)'].sum().sort_values(ascending=False).reset_index()
total_profit.columns = ['BANK', 'TotalProfit']

# Assign tiers: top 3 = Large, next 3 = Medium, bottom 3 = Small
total_profit['Tier'] = pd.qcut(total_profit['TotalProfit'], q=3, labels=['Small', 'Medium', 'Large'])
# Note: qcut with 3 groups will split into equal-sized groups (3 banks each if 9 banks)

print(total_profit)

# Merge tier info back to main dataframe
df = df.merge(total_profit[['BANK', 'Tier']], on='BANK', how='left')

# %%
# Verify tier assignment
print(df.groupby('Tier')['BANK'].unique())

# %%
# Calculate average Net NPA per year per tier
tier_npa = df.groupby(['StartYear', 'Tier'])['Net NPA (%)'].mean().reset_index()

# Pivot for easier plotting
pivot_df = tier_npa.pivot(index='StartYear', columns='Tier', values='Net NPA (%)').fillna(0)
# Ensure columns are in order Large, Medium, Small
pivot_df = pivot_df[['Large', 'Medium', 'Small']]

print(pivot_df.head())

# %%
# Plotting
plt.figure(figsize=(12, 6))
for tier in ['Large', 'Medium', 'Small']:
    plt.plot(pivot_df.index, pivot_df[tier], marker='o', label=tier)

plt.xlabel('Year')
plt.ylabel('Average Net NPA (%)')
plt.title('Tier-wise Net NPA Across Years')
plt.legend(title='Tier')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45)

# Ensure the output directory exists
output_dir = '/charts/systemic'
os.makedirs(output_dir, exist_ok=True)

# Save the chart
output_path = os.path.join(output_dir, '32_terirwise_net_npa.png')
plt.tight_layout()
plt.savefig(output_path, dpi=300)
plt.show()

print(f"Chart saved to {output_path}")